In [ ]:
from pathlib import Path

import iplotx as ipx
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import seaborn as sns

from climate_attitudes import configure_mpl
from climate_attitudes.correlation import Correlation
from climate_attitudes.dataset import Dataset
from climate_attitudes.datasets.reduced import GROUPS as column_groups
from climate_attitudes.settings import Config

FONT_PATH = Path("../fonts")
configure_mpl(FONT_PATH)

plt.rc("figure", dpi=150)

In [ ]:
def get_communities(
    df,
    kind: Correlation,
    threshold: float = 0.7,
    n: int = 1000,
    print_output: bool = False,
):
    columns = np.asarray(df.columns[2:])
    corr = kind.calculate(df)
    corr[np.diag_indices_from(corr)] = 0
    corr = abs(corr)
    G = nx.from_numpy_array(corr)

    freqs = {}
    for _ in range(n):
        # for cluster in sorted(nx.community.louvain_communities(G)):
        for cluster in sorted(nx.community.girvan_newman(G)):
            _cluster = frozenset(cluster)

            freqs[_cluster] = freqs.get(_cluster, 0) + 1

    updated_freqs = {k: v for k, v in freqs.items()}
    for cluster in freqs:
        for other in freqs:
            if cluster < other:
                updated_freqs[cluster] += freqs[other]

    grouping = []
    for cluster, count in list(sorted(updated_freqs.items(), key=lambda x: -x[1])):
        prop = count / n
        if prop < threshold:
            break
        colnames = columns[list(cluster)]
        grouping.append(set(cluster))
        if print_output:
            print(f"p={prop}: {colnames}")
    return grouping


def plot_community_connections(
    df,
    kinds: list[Correlation],
    prevalence_threshold: float = 0.7,
    exclude_subsets: bool = False,
):
    comms = dict()
    for kind in kinds:
        for comm in get_communities(df, kind, threshold=prevalence_threshold):
            comm = frozenset(comm)
            if comm not in comms:
                comms[comm] = []
            comms[comm].append(str(kind))

            # if comm in comms:
            #     continue
            # comms.append(comm)

    # Remove any subsets from array --- all info already captured by superset
    comms_list = [set(comm) for comm in comms]
    if exclude_subsets:
        to_remove = []
        for i, comm in enumerate(comms_list):
            for other in comms_list:
                if comm < other:
                    to_remove.append(i)
        comms_list = [comm for i, comm in enumerate(comms_list) if i not in to_remove]

    adj = np.zeros((len(comms_list), len(comms_list)), dtype=int)
    for i, comm in enumerate(comms_list):
        for j, other in enumerate(comms_list[:i]):
            if comm & other:
                adj[i, j] = adj[j, i] = 1

    G = nx.from_numpy_array(adj)
    layout = nx.forceatlas2_layout(G, gravity=2)

    columns = np.asarray(resp.columns[2:])
    node_labels = [
        ",".join([columns[col_idx] for col_idx in comm]) for comm in comms_list
    ]

    fig, ax = plt.subplots(figsize=(10, 10), constrained_layout=True)
    with ipx.style.context(
        [
            "hollow",
            {
                "vertex": {
                    "linewidth": 1,
                },
                "edge": {
                    "linewidth": 1,
                },
            },
        ]
    ):
        _ = ipx.network(
            G,
            layout=layout,
            node_labels=node_labels,
            margins=0.3,
            ax=ax,
        )[0]

    return comms

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config, name="reduced_no_imputation", with_imputation=False)
resp = dataset.response.collect()
indices = dataset.indices.collect()

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config, name="reduced", with_imputation=True)
resp = dataset.response.collect()
indices = dataset.indices.collect()

In [ ]:
resp.select("cc10")

In [ ]:
dataset.index_result["politics"].explained_variance_ratio_

In [ ]:
explained_variance_prop = []
group_names = []
for name, pca_result in dataset.index_result.items():
    group_names.append(name)
    explained_variance_prop.append(pca_result.explained_variance_ratio_.item())
sns.barplot(x=group_names, y=explained_variance_prop)

In [ ]:
plot_data = indices.drop("participant_id", "wave")
n_vars = plot_data.shape[1]
col_wrap = 3
n_rows = ((n_vars - 1) // col_wrap) + 1

fig, axes = plt.subplots(
    nrows=n_rows, ncols=col_wrap, figsize=(6, 1.5 * n_rows), constrained_layout=True
)

for col, ax in zip(plot_data.columns, axes.flatten(), strict=False):
    sns.histplot(plot_data, x=col, ax=ax, stat="probability")
    ax.set_title(col)
    ax.set_xlabel(None)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for ax in axes[:, 1:].flatten():
    ax.set_ylabel(None)

In [ ]:
group_lookup = {
    "politics": "Politics",
    "extreme_weather": "Extreme weather",
    # "cc_behaviour_change": "CC Behaviour Change",
    "cc_rational": "CC Rational",
    "cc_impacts": "CC Impacts",
    "cc_policy": "CC Policy",
}

fig, axes = plt.subplots(
    nrows=len(group_lookup), figsize=(6, 2 * len(group_lookup)), constrained_layout=True
)

for col, ax in zip(group_lookup, axes, strict=True):
    pca = dataset.index_result[col]
    sns.heatmap(
        (pca.components_.T * np.sqrt(pca.explained_variance_)).T,
        center=0,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        square=True,
        cbar=None,
        vmin=-1,
        vmax=1,
        # cmap=DIVERGING_CMAP,
        ax=ax,
    )

    # Y-axis ticks not meaninful since only using one PCA component
    ax.set_yticks([])
    ax.set_ylabel(group_lookup[col])

    constituent_cols = column_groups[group_lookup[col]]
    ax.set_xticks(
        np.arange(len(constituent_cols)) + 0.5,
        constituent_cols,
        rotation=30,
        horizontalalignment="right",
    )

    # Left-align heatmaps
    ax.set_anchor("W")

In [ ]:
def plot_corr_network(df, corr, threshold: float = 0.05, directed: bool = False):
    fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)

    DIVERGING_CMAP = sns.diverging_palette(20, 230, as_cmap=True)

    # Generate a mask for the upper triangle
    if not directed:
        mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    else:
        mask = np.full_like(corr, fill_value=False, dtype=bool)

    # If `mask_below` set, reset square colour below abs value to zero
    mask[abs(corr) < threshold] = True

    # Remove questions where only diagonal is unmasked
    keep_idxes = (2 * mask.shape[1] - (mask.sum(axis=1) + mask.sum(axis=0))) > 2
    corr = corr[keep_idxes][:, keep_idxes]
    mask = mask[keep_idxes][:, keep_idxes]
    node_labels = np.asarray(df.columns)[keep_idxes]

    # ======== Network
    adj = corr
    adj[np.diag_indices_from(adj)] = 0
    adj[mask] = 0
    if directed:
        G = nx.from_numpy_array(adj, create_using=nx.DiGraph)
    else:
        G = nx.from_numpy_array(adj)
    edge_linewidths = {(u, v): z["weight"] * 12 for u, v, z in G.edges(data=True)}
    edge_colours = [z["weight"] for u, v, z in G.edges(data=True)]
    edge_labels = [f"{z['weight']:.2f}" for u, v, z in G.edges(data=True)]
    layout = nx.forceatlas2_layout(G, gravity=0.3, scaling_ratio=5)

    with ipx.style.context(
        [
            "hollow",
            {
                "vertex": {
                    "linewidth": 1,
                },
                "edge": {
                    "color": edge_colours,
                    "alpha": 1,
                    "cmap": DIVERGING_CMAP,
                    "norm": mcolors.Normalize(vmin=-1, vmax=1),
                },
            },
        ]
    ):
        network_artist = ipx.network(
            G,
            layout=layout,
            tension=1,
            edge_labels=edge_labels,
            node_labels=node_labels,
            edge_linewidth=edge_linewidths,
            edge_curved=True,
            # aspect="equal",
            margins=0.3,
            edge_label_bbox=dict(
                edgecolor="black",
                facecolor="white",
                linewidth=0.25,
                boxstyle="round,pad=0.3",
            ),
            edge_label_rotate=True,
            vertex_facecolor="white",
            vertex_zorder=3,
            ax=ax,
        )[0]
        fig.colorbar(
            network_artist.get_edges(),
            shrink=0.6,
            aspect=30,
            ax=ax,
        )

    fig.suptitle("Partial correlation (GLASSO regularisation)")


plt.rc("figure", dpi=200)
corr = Correlation.PARTIAL_GLASSO.calculate(indices, assume_centered=True)
plot_corr_network(indices.drop("participant_id", "wave"), corr)

In [ ]:
plot_community_connections(
    resp,
    [
        Correlation.PEARSON,
        Correlation.PARTIAL,
        Correlation.DISTANCE_CORR,
        Correlation.VAR_CONTEMPORANEOUS,
    ],
    prevalence_threshold=0.9,
    exclude_subsets=True,
);

In [ ]:
comms = plot_community_connections(
    resp,
    [
        Correlation.PEARSON,
        Correlation.PARTIAL,
        Correlation.DISTANCE_CORR,
        Correlation.VAR_CONTEMPORANEOUS,
    ],
    prevalence_threshold=0.9,
)

In [ ]:
revised_comms = {k: v for k, v in comms.items()}
for comm, kinds in revised_comms.items():
    for other, other_kinds in revised_comms.items():
        if comm == other:
            continue
        elif comm < other:
            kinds.extend([kind for kind in other_kinds if kind not in kinds])

to_remove = []
for comm, kinds in revised_comms.items():
    for other, other_kinds in revised_comms.items():
        if comm == other:
            continue
        elif comm > other and sorted(kinds) == sorted(other_kinds):
            to_remove.append(comm)
revised_comms = {k: v for k, v in revised_comms.items() if k not in to_remove}

columns = np.asarray(resp.columns[2:])
revised_comms = {
    ",".join([columns[idx] for idx in comm]): kinds
    for comm, kinds in revised_comms.items()
}
revised_comms = {k: v for k, v in revised_comms.items() if len(v) > 1}

In [ ]:
revised_comms

In [ ]:
plot_community_connections(
    resp,
    [
        Correlation.PEARSON,
        Correlation.PARTIAL,
        Correlation.DISTANCE_CORR,
        Correlation.VAR_CONTEMPORANEOUS,
    ],
)

In [ ]:
plot_community_connections(resp, Correlation.PEARSON)

In [ ]:
plot_community_connections(resp, Correlation.PARTIAL)

In [ ]:
plot_community_connections(resp, Correlation.DISTANCE_CORR)

In [ ]:
plot_community_connections(resp, Correlation.VAR_CONTEMPORANEOUS)

Pearson

In [ ]:
# plot_groupings(resp, Correlation.PEARSON, corr_threshold=0.1)

In [ ]:
# plot_groupings(resp, Correlation.PARTIAL)

In [ ]:
# plot_groupings(resp, Correlation.DISTANCE_CORR, corr_threshold=0.35)

In [ ]:
# plot_groupings(resp, Correlation.VAR_CONTEMPORANEOUS)

In [ ]:
corr = Correlation.PEARSON.calculate(resp)
corr[np.diag_indices_from(corr)] = 0
corr = abs(corr)
G = nx.from_numpy_array(corr)

n = 1000
freqs = {}
cols = np.asarray(resp.columns[2:])
for _ in range(n):
    for cluster in sorted(nx.community.girvan_newman(G)):
        _cluster = frozenset(cluster)

        freqs[_cluster] = freqs.get(_cluster, 0) + 1

updated_freqs = {k: v for k, v in freqs.items()}
for cluster in freqs:
    for other in freqs:
        if cluster < other:
            updated_freqs[cluster] += freqs[other]

for cluster, count in list(sorted(updated_freqs.items(), key=lambda x: -x[1]))[:100]:
    prop = count / n
    if prop < 0.7:
        break
    colnames = cols[list(cluster)]
    print(f"p={prop}: {colnames}")

Partial

In [ ]:
corr = Correlation.PARTIAL.calculate(resp)
corr[np.diag_indices_from(corr)] = 0
corr = abs(corr)
G = nx.from_numpy_array(corr)

n = 1000
freqs = {}
cols = np.asarray(resp.columns[2:])
for _ in range(n):
    for cluster in sorted(nx.community.louvain_communities(G)):
        _cluster = frozenset(cluster)

        freqs[_cluster] = freqs.get(_cluster, 0) + 1

updated_freqs = {k: v for k, v in freqs.items()}
for cluster in freqs:
    for other in freqs:
        if cluster < other:
            updated_freqs[cluster] += freqs[other]

grouping = []
for cluster, count in list(sorted(updated_freqs.items(), key=lambda x: -x[1]))[:100]:
    prop = count / n
    if prop < 0.7:
        break
    colnames = cols[list(cluster)]
    grouping.append(set(cluster))
    print(f"p={prop}: {colnames}")

In [ ]:
grouping

Distance

In [ ]:
corr = Correlation.DISTANCE_CORR.calculate(resp)
corr[np.diag_indices_from(corr)] = 0
corr = abs(corr)
G = nx.from_numpy_array(corr)

n = 1000
freqs = {}
cols = np.asarray(resp.columns[2:])
for _ in range(n):
    for cluster in sorted(nx.community.louvain_communities(G)):
        _cluster = frozenset(cluster)

        freqs[_cluster] = freqs.get(_cluster, 0) + 1

updated_freqs = {k: v for k, v in freqs.items()}
for cluster in freqs:
    for other in freqs:
        if cluster < other:
            updated_freqs[cluster] += freqs[other]

for cluster, count in list(sorted(updated_freqs.items(), key=lambda x: -x[1]))[:100]:
    prop = count / n
    if prop < 0.7:
        break
    colnames = cols[list(cluster)]
    print(f"p={prop}: {colnames}")

Contemporaneous

In [ ]:
corr = Correlation.VAR_CONTEMPORANEOUS.calculate(resp)
corr[np.diag_indices_from(corr)] = 0
corr = abs(corr)
G = nx.from_numpy_array(corr)

n = 1000
freqs = {}
cols = np.asarray(resp.columns[2:])
for _ in range(n):
    for cluster in sorted(nx.community.louvain_communities(G)):
        _cluster = frozenset(cluster)

        freqs[_cluster] = freqs.get(_cluster, 0) + 1

updated_freqs = {k: v for k, v in freqs.items()}
for cluster in freqs:
    for other in freqs:
        if cluster < other:
            updated_freqs[cluster] += freqs[other]

for cluster, count in list(sorted(updated_freqs.items(), key=lambda x: -x[1]))[:100]:
    prop = count / n
    if prop < 0.7:
        break
    colnames = cols[list(cluster)]
    print(f"p={prop}: {colnames}")

In [ ]:
sorted(updated_freqs.items(), key=lambda x: -x[1])

In [ ]:
for i, comm in enumerate(nx.community.girvan_newman(G)):
    if i == 2:
        cols = np.asarray(resp.columns[2:])
        for cluster in comm:
            print(cols[list(cluster)])
        # print(comm)
        break

In [ ]:
# Source - https://stackoverflow.com/a/59827653
# Posted by Giora Simchoni, modified by community. See post 'Timeline' for change
#   history
# Retrieved 2026-03-23, License - CC BY-SA 4.0


from itertools import chain, combinations

from scipy.cluster.hierarchy import dendrogram

# get simulated Graph() and Girvan-Newman communities list
# G = nx.path_graph(10)
communities = list(nx.community.girvan_newman(G))

# building initial dict of node_id to each possible subset:
node_id = 0
init_node2community_dict = {node_id: communities[0][0].union(communities[0][1])}
for comm in communities:
    for subset in list(comm):
        if subset not in init_node2community_dict.values():
            node_id += 1
            init_node2community_dict[node_id] = subset

# turning this dictionary to the desired format in @mdml's answer
node_id_to_children = {e: [] for e in init_node2community_dict}
for node_id1, node_id2 in combinations(init_node2community_dict.keys(), 2):
    for node_id_parent, group in init_node2community_dict.items():
        if len(
            init_node2community_dict[node_id1].intersection(
                init_node2community_dict[node_id2]
            )
        ) == 0 and group == init_node2community_dict[node_id1].union(
            init_node2community_dict[node_id2]
        ):
            node_id_to_children[node_id_parent].append(node_id1)
            node_id_to_children[node_id_parent].append(node_id2)

# also recording node_labels dict for the correct label for dendrogram leaves
node_labels = dict()
for node_id, group in init_node2community_dict.items():
    if len(group) == 1:
        node_labels[node_id] = list(group)[0]
    else:
        node_labels[node_id] = ""

# also needing a subset to rank dict to later know within all k-length merges which
# came first
subset_rank_dict = dict()
rank = 0
for e in communities[::-1]:
    for p in list(e):
        if tuple(p) not in subset_rank_dict:
            subset_rank_dict[tuple(sorted(p))] = rank
            rank += 1
subset_rank_dict[tuple(sorted(chain.from_iterable(communities[-1])))] = rank


# my function to get a merge height so that it is unique (probably not that efficient)
def get_merge_height(sub):
    sub_tuple = tuple(sorted([node_labels[i] for i in sub]))
    # sub_tuple = tuple(sorted(sub))
    n = len(sub_tuple)
    other_same_len_merges = {k: v for k, v in subset_rank_dict.items() if len(k) == n}
    min_rank, max_rank = (
        min(other_same_len_merges.values()),
        max(other_same_len_merges.values()),
    )
    range = (max_rank - min_rank) if max_rank > min_rank else 1
    return float(len(sub)) + 0.8 * (subset_rank_dict[sub_tuple] - min_rank) / range


# finally using @mdml's magic, slightly modified:
G = nx.DiGraph(node_id_to_children)
nodes = G.nodes()
leaves = set(n for n in nodes if G.out_degree(n) == 0)
inner_nodes = [n for n in nodes if G.out_degree(n) > 0]

# Compute the size of each subtree
subtree = dict((n, [n]) for n in leaves)
for u in inner_nodes:
    children = set()
    node_list = list(node_id_to_children[u])
    while len(node_list) > 0:
        v = node_list.pop(0)
        children.add(v)
        node_list += node_id_to_children[v]
    subtree[u] = sorted(children & leaves)

inner_nodes.sort(
    key=lambda n: len(subtree[n])
)  # <-- order inner nodes ascending by subtree size, root is last

# Construct the linkage matrix
leaves = sorted(leaves)
index = dict((tuple([n]), i) for i, n in enumerate(leaves))
Z = []
k = len(leaves)
for i, n in enumerate(inner_nodes):
    children = node_id_to_children[n]
    x = children[0]
    for y in children[1:]:
        z = tuple(sorted(subtree[x] + subtree[y]))
        i, j = index[tuple(sorted(subtree[x]))], index[tuple(sorted(subtree[y]))]
        try:
            Z.append(
                [i, j, get_merge_height(subtree[n]), len(z)]
            )  # <-- float is required by the dendrogram function
        except:
            print(n)
            raise
        index[z] = k
        subtree[z] = list(z)
        x = z
        k += 1

# dendrogram
plt.figure()
dendrogram(Z, labels=[node_labels[node_id] for node_id in leaves])
plt.savefig("dendrogram.png")

In [ ]:
subtree